In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "README.md").is_file() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "README.md").is_file():
    raise FileNotFoundError("Run this notebook from the project directory or one of its subdirectories.")


# EXP03 — iTransformer + Extended Wave Dynamics + High-Wave Weighted Loss

이 노트북은 물리 정제가 완료된 `train_final_physics.csv`를 입력으로 사용합니다.

### EXP03 주요 개선점
1. **신규 물리 피처 풀 확장**: 파고 모멘텀(`hs_diff_1h`, `hs_diff_3h`), 파랑 에너지(`wave_energy`), 유효 풍응력(`effective_wind_forcing`) 추가
2. **고파랑 가중치 손실함수 (High-Wave Weighted MSE)**: $h_s \ge 1.5\text{m}$ 고파랑 구간에 추가 가중치를 부여하여 `comp_rmse` 집중 최적화
3. **안정적인 48h 슬라이딩 윈도우 & 결측치 완벽 방어**

In [ ]:
# ============================================================
# 0. SETUP & PACKAGES
# ============================================================
!pip -q install optuna

from pathlib import Path
import gc
import json
import math
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================
DATA_PATH = Path(PROJECT_ROOT / "data" / "processed" / "train_final_physics.csv")
TEST_CONTEXT_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_context.parquet")
TEST_INDEX_PATH = Path(PROJECT_ROOT / "data" / "raw" / "test_index.csv")
SUBMISSION_PATH = Path(PROJECT_ROOT / "submission" / "submission_exp03.csv")

EXP_DIR = PROJECT_ROOT / "artifacts" / "experiments" / "exp03"
EXP_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "time"
STATION_COL = "station"

STEP_MINUTES = 10
STEPS_PER_HOUR = 6

INPUT_LEN = 289
LEAD_HOURS = [3, 6, 9, 12, 18, 24]
LEAD_STEPS = [h * STEPS_PER_HOUR for h in LEAD_HOURS]
MAX_LEAD = max(LEAD_STEPS)
N_TARGETS = len(LEAD_STEPS)

TRAIN_RATIO = 0.80

ABLATION_EPOCHS = 12
ABLATION_PATIENCE = 3

N_TRIALS = 30
OPTUNA_EPOCHS = 18
OPTUNA_PATIENCE = 4

FINAL_EPOCHS = 35
FINAL_PATIENCE = 6

NUM_WORKERS = 2 if torch.cuda.is_available() else 0

print("INPUT_LEN:", INPUT_LEN)
print("LEADS:", LEAD_HOURS)


In [ ]:
# ============================================================
# 2. LOAD DATA
# ============================================================
df = pd.read_csv(DATA_PATH, parse_dates=[TIME_COL])
df = df.sort_values([STATION_COL, TIME_COL]).reset_index(drop=True)

if "hs_original_observed" not in df.columns:
    df["hs_original_observed"] = df["hs"].notna().astype(np.int8)

print("shape:", df.shape)
print("stations:", df[STATION_COL].unique())
print("period:", df[TIME_COL].min(), "~", df[TIME_COL].max())
print("Columns:", df.columns.tolist())

display(
    df.groupby(STATION_COL)
    .agg(
        rows=("hs", "size"),
        hs_missing=("hs", lambda s: s.isna().sum()),
    )
)


In [ ]:
# ============================================================
# 3. EXPANDED FEATURE GROUPS
# ============================================================
FEATURE_GROUPS = {
    "BASE_WAVE": ["hs", "tp", "hmax"],
    "WIND_BASIC": ["wspd", "gust", "u_wind", "v_wind"],
    "WAVE_MOMENTUM": ["hs_diff_1h", "hs_diff_3h", "hs_mean_6h", "hs_mean_12h", "hs_max_6h", "hs_max_12h"],
    "WAVE_DYNAMICS": ["wave_steepness", "wave_energy", "effective_wind_forcing", "u_wave", "v_wave"],
    "WIND_HISTORY": ["wspd_mean_6h", "wspd_mean_12h", "gust_max_6h", "gust_max_12h", "gust_minus_wspd"],
    "PRESSURE_TENDENCY": ["caph_change_3h", "caph_change_6h", "caph_change_12h"],
    "ALIGNMENT": ["wind_wave_alignment", "wind_wave_diff"],
    "ATMOS": ["airt", "relh", "caph"],
}

for k, cols in list(FEATURE_GROUPS.items()):
    FEATURE_GROUPS[k] = [c for c in cols if c in df.columns]

print("=== FEATURE GROUPS ===")
for k, cols in FEATURE_GROUPS.items():
    print(k, "->", len(cols), cols)


In [ ]:
# ============================================================
# 4. ABLATION SETS DEFINITION
# ============================================================
def uniq(seq):
    return list(dict.fromkeys(seq))

BASE = FEATURE_GROUPS["BASE_WAVE"]

FEATURE_SETS = {
    "BASE_WAVE": uniq(BASE),
    "BASE_WIND": uniq(BASE + FEATURE_GROUPS["WIND_BASIC"]),
    "WAVE_MOMENTUM_SET": uniq(BASE + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WAVE_MOMENTUM"]),
    "WAVE_DYNAMICS_SET": uniq(BASE + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WAVE_DYNAMICS"]),
    "STORM_DYNAMICS": uniq(BASE + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WAVE_MOMENTUM"] + FEATURE_GROUPS["WAVE_DYNAMICS"] + FEATURE_GROUPS["PRESSURE_TENDENCY"]),
    "FULL_PHYSICS_V3": uniq(
        BASE + FEATURE_GROUPS["WIND_BASIC"] + FEATURE_GROUPS["WAVE_MOMENTUM"] +
        FEATURE_GROUPS["WAVE_DYNAMICS"] + FEATURE_GROUPS["WIND_HISTORY"] +
        FEATURE_GROUPS["PRESSURE_TENDENCY"] + FEATURE_GROUPS["ALIGNMENT"] + FEATURE_GROUPS["ATMOS"]
    )
}

for k, cols in list(FEATURE_SETS.items()):
    FEATURE_SETS[k] = [c for c in cols if c in df.columns]

display(
    pd.DataFrame([
        {
            "feature_set": k,
            "n_features": len(v),
            "features": ", ".join(v),
        }
        for k, v in FEATURE_SETS.items()
    ])
)


In [ ]:
# ============================================================
# 5. SAMPLE BUILDING
# ============================================================
STATION_TO_ID = {station: i for i, station in enumerate(sorted(df[STATION_COL].unique()))}

def build_samples(frame, features, input_len=INPUT_LEN, lead_steps=LEAD_STEPS):
    X_list, y_list, origin_hs_list, origin_time_list, station_list, end_idx_list = [], [], [], [], [], []
    max_lead = max(lead_steps)

    for station, g in frame.groupby(STATION_COL):
        g = g.sort_values(TIME_COL).reset_index(drop=True)
        Xv = g[features].to_numpy(dtype=np.float32)
        hsv = g["hs"].to_numpy(dtype=np.float32)
        obs = g["hs_original_observed"].to_numpy(dtype=np.int8)
        times = g[TIME_COL].to_numpy()

        dt = pd.Series(g[TIME_COL]).diff().dt.total_seconds().div(60).to_numpy()

        for end_idx in range(input_len - 1, len(g) - max_lead):
            start_idx = end_idx - input_len + 1
            local_dt = dt[start_idx + 1:end_idx + 1]
            if len(local_dt) and not np.all(local_dt == STEP_MINUTES):
                continue

            x = Xv[start_idx:end_idx + 1]
            if not np.isfinite(x).all():
                continue

            target_idx = np.asarray([end_idx + s for s in lead_steps], dtype=int)
            y = hsv[target_idx]
            if not np.isfinite(y).all() or not np.all(obs[target_idx] == 1):
                continue

            origin_hs = hsv[end_idx]
            if not np.isfinite(origin_hs):
                continue

            X_list.append(x)
            y_list.append(y)
            origin_hs_list.append(origin_hs)
            origin_time_list.append(times[end_idx])
            station_list.append(STATION_TO_ID[station])
            end_idx_list.append(end_idx)

    if not X_list:
        return {
            "X": np.empty((0, input_len, len(features)), dtype=np.float32),
            "y": np.empty((0, len(lead_steps)), dtype=np.float32),
            "origin_hs": np.empty((0,), dtype=np.float32),
            "origin_time": np.empty((0,), dtype="datetime64[ns]"),
            "station_id": np.empty((0,), dtype=np.int64),
            "end_idx": np.empty((0,), dtype=np.int64),
        }

    return {
        "X": np.asarray(X_list, dtype=np.float32),
        "y": np.asarray(y_list, dtype=np.float32),
        "origin_hs": np.asarray(origin_hs_list, dtype=np.float32),
        "origin_time": np.asarray(origin_time_list),
        "station_id": np.asarray(station_list, dtype=np.int64),
        "end_idx": np.asarray(end_idx_list, dtype=np.int64),
    }

def chronological_split(samples, train_ratio=TRAIN_RATIO):
    times = pd.to_datetime(samples["origin_time"])
    cutoff = pd.Series(times).quantile(train_ratio)
    train_mask = times <= cutoff
    valid_mask = times > cutoff
    train = {k: v[train_mask] for k, v in samples.items()}
    valid = {k: v[valid_mask] for k, v in samples.items()}
    return train, valid, pd.Timestamp(cutoff)

def scale_samples(train, valid):
    n_features = train["X"].shape[-1]
    scaler = StandardScaler()
    scaler.fit(train["X"].reshape(-1, n_features))
    train_scaled = dict(train)
    valid_scaled = dict(valid)
    train_scaled["X"] = scaler.transform(train["X"].reshape(-1, n_features)).reshape(train["X"].shape).astype(np.float32)
    valid_scaled["X"] = scaler.transform(valid["X"].reshape(-1, n_features)).reshape(valid["X"].shape).astype(np.float32)
    return train_scaled, valid_scaled, scaler


In [ ]:
# ============================================================
# 6. USABLE WINDOW REPORT
# ============================================================
window_records = []
for feature_name, features in FEATURE_SETS.items():
    samples = build_samples(df, features)
    station_ids = samples["station_id"]
    record = {
        "feature_set": feature_name,
        "n_features": len(features),
        "usable_total": len(samples["y"]),
        "origin_hs_ge_1.5": int((samples["origin_hs"] >= 1.5).sum()),
    }
    for station, sid in STATION_TO_ID.items():
        record[f"{station}_n"] = int((station_ids == sid).sum())
    window_records.append(record)

window_report = pd.DataFrame(window_records)
display(window_report.sort_values("usable_total", ascending=False))
window_report.to_csv(EXP_DIR / "usable_windows.csv", index=False)


In [ ]:
# ============================================================
# 7. PYTORCH DATASET
# ============================================================
class WaveDataset(Dataset):
    def __init__(self, samples):
        self.X = torch.from_numpy(samples["X"]).float()
        self.y = torch.from_numpy(samples["y"]).float()
        self.origin_hs = torch.from_numpy(samples["origin_hs"]).float()
        self.station_id = torch.from_numpy(samples["station_id"]).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.origin_hs[idx], self.station_id[idx]

def make_loader(samples, batch_size=128, shuffle=False):
    ds = WaveDataset(samples)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )


In [ ]:
# ============================================================
# 8. iTransformer MODEL DEFINITION
# ============================================================
class ITransformer(nn.Module):
    def __init__(self, seq_len, n_features, pred_len, d_model=64, n_heads=2, e_layers=3, dropout=0.25, d_ff=None):
        super().__init__()
        self.seq_len = seq_len
        self.n_features = n_features
        self.pred_len = pred_len
        if d_ff is None:
            d_ff = d_model * 4

        self.value_embedding = nn.Linear(seq_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=e_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, pred_len),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.value_embedding(x)
        x = self.encoder(x)
        x = self.norm(x)
        out = self.head(x)
        return out


In [ ]:
# ============================================================
# 9. METRICS & LOSS FUNCTION
# ============================================================
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def lead_rmse(y_true, y_pred):
    return {f"rmse_{h}h": rmse(y_true[:, i], y_pred[:, i]) for i, h in enumerate(LEAD_HOURS)}

def competition_like_rmse(y_true, y_pred, origin_hs):
    mask = origin_hs >= 1.5
    if mask.sum() == 0:
        return np.nan, 0
    return rmse(y_true[mask], y_pred[mask]), int(mask.sum())

def select_78h_separated_indices(times, station_ids, origin_hs, min_hours=78):
    selected = []
    times = pd.to_datetime(times)
    for sid in np.unique(station_ids):
        idx = np.where((station_ids == sid) & (origin_hs >= 1.5))[0]
        idx = idx[np.argsort(times[idx])]
        last_time = None
        for i in idx:
            t = times[i]
            if last_time is None or (t - last_time) >= pd.Timedelta(hours=min_hours):
                selected.append(i)
                last_time = t
    return np.asarray(selected, dtype=int)

def exact_competition_rmse(y_true, y_pred, samples):
    idx = select_78h_separated_indices(samples["origin_time"], samples["station_id"], samples["origin_hs"], min_hours=78)
    if len(idx) == 0:
        return np.nan, 0
    return rmse(y_true[idx], y_pred[idx]), len(idx)

class HighWaveWeightedLoss(nn.Module):
    def __init__(self, threshold=1.5, weight=2.0):
        super().__init__()
        self.threshold = threshold
        self.weight = weight

    def forward(self, pred, target, origin_hs):
        diff_sq = (pred - target) ** 2
        weights = torch.where(origin_hs >= self.threshold, self.weight, 1.0).unsqueeze(-1)
        return torch.mean(weights * diff_sq)


In [ ]:
# ============================================================
# 10. TRAIN & EVAL FUNCTIONS
# ============================================================
def evaluate_model(model, loader):
    model.eval()
    preds, ys, origins = [], [], []
    with torch.no_grad():
        for X, y, origin_hs, _ in loader:
            X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            pred = model(X)
            preds.append(pred.cpu().numpy())
            ys.append(y.cpu().numpy())
            origins.append(origin_hs.numpy())

    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    origin_hs = np.concatenate(origins)

    overall = rmse(y_true, y_pred)
    comp, comp_n = competition_like_rmse(y_true, y_pred, origin_hs)
    result = {
        "overall_rmse": overall,
        "comp_rmse": comp,
        "comp_valid_n": comp_n,
        **lead_rmse(y_true, y_pred),
    }
    return result, y_true, y_pred

def train_one_model(train_samples, valid_samples, params, max_epochs, patience, verbose=True):
    seed_everything(SEED)
    batch_size = params.get("batch_size", 128)
    train_loader = make_loader(train_samples, batch_size=batch_size, shuffle=True)
    valid_loader = make_loader(valid_samples, batch_size=batch_size, shuffle=False)

    model = ITransformer(
        seq_len=INPUT_LEN,
        n_features=train_samples["X"].shape[-1],
        pred_len=N_TARGETS,
        d_model=params["d_model"],
        n_heads=params["n_heads"],
        e_layers=params["e_layers"],
        dropout=params["dropout"],
        d_ff=params.get("d_ff", params["d_model"] * 4),
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    criterion = HighWaveWeightedLoss(threshold=1.5, weight=2.0)

    best_state = None
    best_score = np.inf
    wait = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for X, y, origin_hs, _ in train_loader:
            X, y, origin_hs = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True), origin_hs.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            pred = model(X)
            loss = criterion(pred, y, origin_hs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())

        metrics, _, _ = evaluate_model(model, valid_loader)
        score = metrics["comp_rmse"]
        if not np.isfinite(score):
            score = metrics["overall_rmse"]

        history.append({"epoch": epoch, "train_loss": float(np.mean(train_losses)), **metrics})
        if verbose:
            print(f"Epoch {epoch:02d} | train={np.mean(train_losses):.5f} | overall={metrics['overall_rmse']:.5f} | comp={metrics['comp_rmse']:.5f}")

        if score < best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_metrics, y_true, y_pred = evaluate_model(model, valid_loader)
    return model, final_metrics, pd.DataFrame(history), y_true, y_pred


In [ ]:
# ============================================================
# 11. FEATURE ABLATION STUDY
# ============================================================
FIXED_PARAMS = {
    "d_model": 128,
    "n_heads": 4,
    "e_layers": 3,
    "dropout": 0.20,
    "lr": 3e-5,
    "weight_decay": 1e-5,
    "batch_size": 128,
}

ablation_records = []
for feature_name, features in FEATURE_SETS.items():
    print("\n" + "=" * 90)
    print("ABLATION:", feature_name, "|", len(features), "features")
    samples = build_samples(df, features)
    if len(samples["y"]) < 1000:
        continue

    train_s, valid_s, cutoff = chronological_split(samples)
    train_s, valid_s, scaler = scale_samples(train_s, valid_s)
    model, metrics, history, y_true, y_pred = train_one_model(
        train_s, valid_s, params=FIXED_PARAMS, max_epochs=ABLATION_EPOCHS, patience=ABLATION_PATIENCE, verbose=False
    )
    exact_comp, exact_n = exact_competition_rmse(y_true, y_pred, valid_s)
    record = {
        "feature_set": feature_name,
        "n_features": len(features),
        "train_n": len(train_s["y"]),
        "valid_n": len(valid_s["y"]),
        "cutoff": cutoff,
        **metrics,
        "exact_78h_comp_rmse": exact_comp,
        "exact_78h_n": exact_n,
    }
    ablation_records.append(record)
    print(pd.Series(record))
    del model, samples, train_s, valid_s, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

ablation_df = pd.DataFrame(ablation_records)
display(ablation_df.sort_values(["comp_rmse", "overall_rmse"], ascending=True))
ablation_df.to_csv(EXP_DIR / "feature_ablation.csv", index=False)


In [ ]:
# ============================================================
# 12. CHOOSE BEST FEATURE SET
# ============================================================
assert len(ablation_df) > 0
max_comp_n = ablation_df["comp_valid_n"].max()
eligible = ablation_df[ablation_df["comp_valid_n"] >= 0.80 * max_comp_n].copy()
eligible = eligible.sort_values(["comp_rmse", "overall_rmse"])

BEST_FEATURE_NAME = eligible.iloc[0]["feature_set"]
BEST_FEATURES = FEATURE_SETS[BEST_FEATURE_NAME]

print("BEST FEATURE SET:", BEST_FEATURE_NAME)
print("N FEATURES:", len(BEST_FEATURES))
print(BEST_FEATURES)
display(eligible)


In [ ]:
# ============================================================
# 13. PREPARE FIXED DATA FOR OPTUNA
# ============================================================
best_samples = build_samples(df, BEST_FEATURES)
train_samples, valid_samples, split_cutoff = chronological_split(best_samples)
train_samples, valid_samples, best_scaler = scale_samples(train_samples, valid_samples)

print("split cutoff:", split_cutoff)
print("train:", len(train_samples["y"]))
print("valid:", len(valid_samples["y"]))
print("high-wave valid:", int((valid_samples["origin_hs"] >= 1.5).sum()))


In [ ]:
# ============================================================
# 14. OPTUNA HYPERPARAMETER TUNING
# ============================================================
def objective(trial):
    d_model = trial.suggest_categorical("d_model", [64, 128, 256])
    valid_heads = [h for h in [2, 4, 8] if d_model % h == 0]
    params = {
        "d_model": d_model,
        "n_heads": trial.suggest_categorical("n_heads", valid_heads),
        "e_layers": trial.suggest_int("e_layers", 2, 4),
        "dropout": trial.suggest_float("dropout", 0.05, 0.35),
        "lr": trial.suggest_float("lr", 5e-6, 1e-4, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [64, 128, 256]),
    }
    try:
        model, metrics, _, _, _ = train_one_model(
            train_samples, valid_samples, params=params, max_epochs=OPTUNA_EPOCHS, patience=OPTUNA_PATIENCE, verbose=False
        )
        trial.set_user_attr("overall_rmse", metrics["overall_rmse"])
        trial.set_user_attr("comp_valid_n", metrics["comp_valid_n"])
        score = metrics["comp_rmse"]
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return score
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            raise optuna.TrialPruned("CUDA OOM")
        raise

study = optuna.create_study(direction="minimize", sampler=TPESampler(seed=SEED), study_name="exp03_itransformer")
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("BEST VALUE:", study.best_value)
print("BEST PARAMS:", study.best_params)


In [ ]:
# ============================================================
# 15. OPTUNA RESULTS SUMMARY
# ============================================================
trials_df = study.trials_dataframe()
display(trials_df.sort_values("value").head(15))
trials_df.to_csv(EXP_DIR / "optuna_trials.csv", index=False)
with open(EXP_DIR / "best_params.json", "w", encoding="utf-8") as f:
    json.dump(study.best_params, f, indent=2, ensure_ascii=False)


In [ ]:
# ============================================================
# 16. FINAL MODEL RETRAINING
# ============================================================
BEST_PARAMS = dict(study.best_params)
final_model, final_metrics, final_history, y_true, y_pred = train_one_model(
    train_samples, valid_samples, params=BEST_PARAMS, max_epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE, verbose=True
)

exact_comp_rmse, exact_comp_n = exact_competition_rmse(y_true, y_pred, valid_samples)
final_metrics["exact_78h_comp_rmse"] = exact_comp_rmse
final_metrics["exact_78h_n"] = exact_comp_n

print("\n===== FINAL EXP03 METRICS =====")
display(pd.Series(final_metrics).to_frame("value"))
final_history.to_csv(EXP_DIR / "final_history.csv", index=False)
pd.DataFrame([final_metrics]).to_csv(EXP_DIR / "final_metrics.csv", index=False)


In [ ]:
# ============================================================
# 17. LEAD-WISE RESULT PLOT
# ============================================================
lead_result = pd.DataFrame({
    "lead_hour": LEAD_HOURS,
    "rmse": [rmse(y_true[:, i], y_pred[:, i]) for i in range(N_TARGETS)],
})
display(lead_result)
lead_result.to_csv(EXP_DIR / "lead_rmse.csv", index=False)

plt.figure(figsize=(7, 4))
plt.plot(lead_result["lead_hour"], lead_result["rmse"], marker="o", color="#1f77b4", linewidth=2)
plt.xlabel("Forecast lead (hour)")
plt.ylabel("RMSE (m)")
plt.title("EXP03 Lead-wise RMSE")
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# ============================================================
# 18. SAVE MODEL + SCALER + METADATA
# ============================================================
import joblib
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "features": BEST_FEATURES,
        "feature_set_name": BEST_FEATURE_NAME,
        "input_len": INPUT_LEN,
        "lead_hours": LEAD_HOURS,
        "params": BEST_PARAMS,
        "station_to_id": STATION_TO_ID,
        "metrics": final_metrics,
    },
    EXP_DIR / "best_model.pt",
)
joblib.dump(best_scaler, EXP_DIR / "scaler.pkl")

metadata = {
    "data_path": str(DATA_PATH),
    "feature_set_name": BEST_FEATURE_NAME,
    "features": BEST_FEATURES,
    "input_len": INPUT_LEN,
    "lead_hours": LEAD_HOURS,
    "split_cutoff": str(split_cutoff),
    "train_n": int(len(train_samples["y"])),
    "valid_n": int(len(valid_samples["y"])),
    "best_params": BEST_PARAMS,
    "metrics": {k: float(v) if isinstance(v, (float, np.floating)) else int(v) if isinstance(v, (int, np.integer)) else v for k, v in final_metrics.items()},
}
with open(EXP_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print("Saved EXP03 artifacts to:", EXP_DIR.resolve())


In [ ]:
# ============================================================
# 19. TEST INFERENCE + SUBMISSION CREATION (EXP03)
# ============================================================
def add_test_features_exp03(context):
    feature_frames = []
    for case_id, group in context.groupby("case_id", sort=False):
        g = group.sort_values("step_minute").copy()
        st = g["station"].iloc[0]

        # 기본 결측 방어 선형보간
        num_cols = ["hs", "tp", "hmax", "wspd", "gust", "airt", "relh", "caph"]
        g[num_cols] = g[num_cols].interpolate(method="linear", limit_direction="both").ffill().bfill()
        g["wdir"] = g["wdir"].ffill().bfill() % 360.0
        g["wvdir"] = g["wvdir"].ffill().bfill() % 360.0
        g["hmax"] = np.maximum(g["hmax"], g["hs"])
        g["gust"] = np.maximum(g["gust"], g["wspd"])

        # 바람 및 파랑 벡터
        wdir_rad = np.deg2rad(g["wdir"])
        wvdir_rad = np.deg2rad(g["wvdir"])
        g["u_wind"] = g["wspd"] * np.sin(wdir_rad)
        g["v_wind"] = g["wspd"] * np.cos(wdir_rad)
        g["u_wave"] = g["hs"] * np.sin(wvdir_rad)
        g["v_wave"] = g["hs"] * np.cos(wvdir_rad)

        # 바람/파향 일치도
        diff = ((g["wdir"] - g["wvdir"] + 180) % 360) - 180
        g["wind_wave_diff"] = np.abs(diff)
        g["wind_wave_alignment"] = np.cos(np.deg2rad(diff))

        # 롤링 통계량 및 파고 모멘텀
        g["hs_diff_1h"] = g["hs"] - g["hs"].shift(6)
        g["hs_diff_3h"] = g["hs"] - g["hs"].shift(18)
        g["hs_mean_6h"] = g["hs"].rolling(36, min_periods=1).mean()
        g["hs_mean_12h"] = g["hs"].rolling(72, min_periods=1).mean()
        g["hs_max_6h"] = g["hs"].rolling(36, min_periods=1).max()
        g["hs_max_12h"] = g["hs"].rolling(72, min_periods=1).max()

        g["wspd_mean_6h"] = g["wspd"].rolling(36, min_periods=1).mean()
        g["wspd_mean_12h"] = g["wspd"].rolling(72, min_periods=1).mean()
        g["gust_max_6h"] = g["gust"].rolling(36, min_periods=1).max()
        g["gust_max_12h"] = g["gust"].rolling(72, min_periods=1).max()
        g["gust_minus_wspd"] = g["gust"] - g["wspd"]

        # 기압 변화량
        g["caph_change_3h"] = g["caph"] - g["caph"].shift(18)
        g["caph_change_6h"] = g["caph"] - g["caph"].shift(36)
        g["caph_change_12h"] = g["caph"] - g["caph"].shift(72)

        # 파랑 역학
        wl = 1.56 * (g["tp"] ** 2)
        g["wave_steepness"] = g["hs"] / np.maximum(wl, 1.0)
        g["wave_energy"] = g["hs"] ** 2
        g["effective_wind_forcing"] = (g["wspd"] ** 2) * g["wind_wave_alignment"]

        # 초기 경계 결측 ffill/bfill
        g[BEST_FEATURES] = g[BEST_FEATURES].bfill().ffill()
        feature_frames.append(g)

    return pd.concat(feature_frames, ignore_index=True)

test_context = pd.read_parquet(TEST_CONTEXT_PATH)
test_index = pd.read_csv(TEST_INDEX_PATH)
test_features = add_test_features_exp03(test_context)

case_order = test_index["case_id"].drop_duplicates().tolist()
windows = []
for case_id in case_order:
    grp = test_features[test_features["case_id"] == case_id].sort_values("step_minute")
    X = grp[BEST_FEATURES].to_numpy(dtype=np.float32)
    windows.append(X)

X_test_raw = np.stack(windows)
X_test = best_scaler.transform(X_test_raw.reshape(-1, len(BEST_FEATURES))).reshape(X_test_raw.shape).astype(np.float32)

final_model.eval()
with torch.no_grad():
    preds = final_model(torch.from_numpy(X_test).to(DEVICE)).cpu().numpy()

pred_rows = []
for case_id, row in zip(case_order, preds):
    for lead_h, val in zip(LEAD_HOURS, row):
        pred_rows.append({"case_id": case_id, "lead_h": lead_h, "hs_pred": float(val)})

submission = test_index.merge(pd.DataFrame(pred_rows), on=["case_id", "lead_h"], how="left")
submission["hs_pred"] = submission["hs_pred"].clip(lower=0.0, upper=30.0)
submission.to_csv(SUBMISSION_PATH, index=False, encoding="utf-8")

print("=" * 80)
print(f"EXP03 SUBMISSION SAVED: {SUBMISSION_PATH}")
print("=" * 80)
display(submission.head(12))
